# 10 Minute Tour of PyTestLab

This tutorial uses simulation mode and does not connect to hardware.

Welcome to PyTestLab! This short tour will guide you through the basic setup and usage of PyTestLab to interact with your lab instruments.

## 1. Installation

Use Python 3.11 or newer and install PyTestLab in the environment used by this notebook's kernel:

```sh
python -m pip install pytestlab
```

The simulated examples below need no physical instruments or vendor VISA runtime. For VISA hardware, install the appropriate runtime (for example, NI-VISA or Keysight IO Libraries).

## 2. Importing PyTestLab and Connecting to an Instrument

PyTestLab's `AutoInstrument` builds an instrument from a built-in profile key, a local YAML file, or a mapping, and opens its backend on first use.

In [ ]:
from pytestlab import AutoInstrument
from pytestlab.experiments.database import MeasurementDatabase

### Connecting to a Simulated Instrument

This tour uses independent simulation fixtures. No physical instruments, VISA driver, or connected signals are required.

In [ ]:
dmm = AutoInstrument.from_preset("keysight/34470A", simulate=True)
try:
    print(f"Simulated multimeter: {dmm.id()}")
finally:
    dmm.close()

### Connecting to a Real Instrument (Example)

To connect to a real instrument, select its profile and provide its verified VISA address. The illustration below is commented out so it cannot connect when you run the tour.

In [ ]:
# Replace the placeholder with the verified resource for your instrument.
# psu = AutoInstrument.from_preset(
#     "keysight/EDU36311A",
#     address_override="TCPIP0::YOUR_CONFIRMED_HOST::inst0::INSTR",
# )
# try:
#     print(psu.id())
# finally:
#     psu.close()

## 3. Basic Operations using New Facades

PyTestLab now provides intuitive facade objects for common instrument operations, allowing for a chained, more readable syntax. These facade methods are synchronous and execute immediately.

### Power Supply (PSU) Facade Example

In [ ]:
psu = AutoInstrument.from_preset("keysight/EDU36311A", simulate=True)
try:
    print(f"PSU ID: {psu.id()}")
    channel = psu.channel(1)
    channel.set(voltage=3.3, current_limit=0.05).on()
    print(f"Output enabled: {channel.get_output_state()}")
    channel.off()
    print(f"Output enabled after disable: {channel.get_output_state()}")
finally:
    psu.close()

### Waveform Generator (WG) Facade Example

In [ ]:
awg = AutoInstrument.from_preset("keysight/EDU33212A", simulate=True)
try:
    print(f"AWG ID: {awg.id()}")
    channel = awg.channel(1)
    channel.setup_sine(frequency=1e3, amplitude=1.0, offset=0.0).enable()
    print(f"Simulated generator frequency: {awg.get_frequency(1):g} Hz")
    channel.disable()
finally:
    awg.close()

## 4. Performing a Simple Measurement (DMM)

In [ ]:
dmm = AutoInstrument.from_preset("keysight/34470A", simulate=True)
try:
    reading = dmm.measure("dc_voltage", range_val=10, uncertainty=False)
    print(f"Nominal simulated voltage: {float(reading):g} {reading.units}")
finally:
    dmm.close()

## 5. Storing and Retrieving Results with MeasurementDatabase

In [ ]:
with MeasurementDatabase("my_lab_data.db") as database:
    codename = database.store_measurement(None, reading, notes="Synthetic tour example")
    saved = database.retrieve_measurement(codename)
    print(f"{codename}: {float(saved):g} {saved.units}")

## 6. Next Steps

This tour covered the basics of installing PyTestLab, connecting to instruments (simulated and real), using the new facades for common operations, and storing/retrieving data.

From here, you can explore:
- Detailed documentation for specific instruments.
- Advanced features like experiment management and complex sweeps.
- Creating your own instrument configuration files.
- Integrating PyTestLab into your automated test scripts.